# 公司因子回测插件使用教程

这份 Notebook 展示如何调用同目录下的 `func.py` 和 `BacktestEngine.py`。核心流程是：准备数据 → 测原始变量 → 测单变量衍生信号 → 做增量测试 → 构建宽口径综合分数。

所有回测都可以返回 Plotly 图；集中导出时可独立控制 HTML 和 PNG。图中包含 Top、Worst、Benchmark，以及 Top/Bench、Worst/Bench、Top/Worst 三条相对曲线。

## 三个主要入口

- `test_raw_variables()`：直接测试一组已有字段，不使用方向或信号配置。Top 是原始值最高组，Worst 是原始值最低组。
- `test_unitary_signals()`：对每个变量分别测试 level、pct、diff。它使用 `higher_is_better` 和 `denominator`，但不使用 `use_*` 和 `weight_*`。
- `test_incremental_signals()`：把候选变量逐个加入基线。候选变量实际使用哪些维度及权重，由 `candidate_config` 决定。

`calculate_composite_score()` 会严格按照配置中的 `use_level/use_pct/use_diff` 和权重构建综合分数，并把衍生列保留在 `screen`。

## 1. 加载插件

In [ ]:
from pathlib import Path
import importlib
import sys
import pandas as pd

PLUGIN_DIR = Path(r"C:\Users\jingx\Downloads\公司回测插件")
if str(PLUGIN_DIR) not in sys.path:
    sys.path.insert(0, str(PLUGIN_DIR))

import BacktestEngine
import func

importlib.reload(BacktestEngine)
importlib.reload(func)

from func import (
    RECOMMENDED_PERIOD_BREAKPOINTS,
    build_periods_from_breakpoints,
    calculate_composite_score,
    calculate_performance_ratios,
    combine_backtest_performances,
    compare_backtest_results,
    export_backtest_results,
    plot_performance_comparison,
    reconstruct_backtest_analysis,
    run_top_worst_backtest,
    test_composite_signal,
    test_incremental_signals,
    test_raw_variables,
    test_unitary_signals,
)

print("插件加载完成")

## 2. 准备数据

运行后续单元格前，需要准备：

- `screen`：股票横截面 DataFrame，至少包含 `ISIN`、`Date`、行业、国家以及待测变量；同时需要满足现有 `PtfBuilder` 对 SEDOL、市值、Benchmark 权重等字段的要求。
- `returns`：回测收益率 DataFrame，格式与现有 `BacktestEngine` 一致。
- `list_noire_path`：黑名单 Excel 文件路径；当前示例按要求设为 `None`。

下面直接读取插件 `data` 目录中的真实 parquet 文件。

In [ ]:
DATA_DIR = PLUGIN_DIR / "data"
SCREEN_PATH = DATA_DIR / "screen_aggregate.parquet"
RETURNS_PATH = DATA_DIR / "returns.parquet"

screen = pd.read_parquet(SCREEN_PATH)
returns = pd.read_parquet(RETURNS_PATH)
list_noire_path = None


In [ ]:
missing_objects = [name for name in ("screen", "returns", "list_noire_path") if name not in globals()]
if missing_objects:
    raise RuntimeError(f"请先准备这些对象：{missing_objects}")

required_columns = {
    "Date",
    " Benchmark ICB Supersector ",
    "Exchange Country Region",
}
missing_columns = sorted(required_columns - set(screen.columns))
if "ISIN" not in screen.columns and screen.index.name != "ISIN":
    missing_columns.append("ISIN")
if missing_columns:
    raise KeyError(f"screen 缺少基础字段：{missing_columns}")

screen["Date"] = pd.to_datetime(screen["Date"])
print(f"screen: {screen.shape}, returns: {returns.shape}")

## 3. 在 Notebook 中定义信号配置

下面只是格式示例。请根据 `screen` 中真实存在的列增删变量。

配置规则：

- `higher_is_better`：高值是否代表更好的方向。
- `denominator`：绝对值变量先除以该字段，例如 R&D / Sales。
- `use_level/use_pct/use_diff`：综合分数和增量测试实际使用哪些维度。
- `weight_*`：对应维度在综合分数中的权重。

In [ ]:
WIDE_CONFIG = {
    "Revenue 5Y CAGR": {
        "higher_is_better": True,
        "use_level": True, "weight_level": 1.0,
        "use_pct": True, "weight_pct": 1.0,
        "use_diff": False, "weight_diff": 0.0,
    },
    "FCF Conversion": {
        "higher_is_better": True,
        "use_level": True, "weight_level": 1.0,
        "use_pct": False, "weight_pct": 0.0,
        "use_diff": True, "weight_diff": 1.0,
    },
    "Net Debt to Ebit": {
        "higher_is_better": False,
        "use_level": True, "weight_level": 1.0,
        "use_pct": False, "weight_pct": 0.0,
        "use_diff": True, "weight_diff": 1.0,
    },
    "R&D Expense CIQ": {
        "higher_is_better": True,
        "denominator": "Sales",
        "use_level": True, "weight_level": 1.0,
        "use_pct": False, "weight_pct": 0.0,
        "use_diff": False, "weight_diff": 0.0,
    },
}

available_config = {name: options for name, options in WIDE_CONFIG.items() if name in screen.columns}
missing_config = sorted(set(WIDE_CONFIG) - set(available_config))
print(f"可用配置变量：{list(available_config)}")
print(f"screen 中不存在、将被跳过：{missing_config}")

## 4. 最简单的用法：直接测试原始变量

这一步不需要信号配置。每个变量生成一张 Top/Worst/Benchmark 对比图。Top 始终代表原始数值最高组，不代表经济含义上一定更好。

In [ ]:
RAW_VARIABLES = [
    "Revenue 5Y CAGR",
    "FCF Conversion",
    "Net Debt to Ebit",
]

raw_results = test_raw_variables(
    screen=screen,
    returns=returns,
    variables=RAW_VARIABLES,
    list_noire_path=list_noire_path,
    percentile=0.13,
    show_plot=True,
)

In [ ]:
raw_summary = pd.DataFrame([
    {
        "Variable": name,
        "Robust Score": result.get("robust_score"),
        "Top/Bench": result.get("top_bench_ratio"),
        "Top/Worst": result.get("top_worst_ratio"),
    }
    for name, result in raw_results.items()
]).sort_values("Robust Score", ascending=False)
raw_summary

## 5. 单变量衍生信号测试

默认对每个变量分别测试 level、pct、diff，因此一个变量最多生成三张图。这里不会读取配置中的 `use_*` 和权重，但会读取方向和分母。

如果只想测试某些维度，可以设置 `dimensions=("pct",)` 或 `dimensions=("level", "diff")`。

In [ ]:
unitary_results = test_unitary_signals(
    screen=screen,
    returns=returns,
    signal_config=available_config,
    list_noire_path=list_noire_path,
    dimensions=("level", "pct", "diff"),
    percentile=0.13,
    show_plot=True,
)

In [ ]:
derived_columns = [
    column for column in screen.columns
    if "__over__" in column or column.endswith("__pct") or column.endswith("__diff")
]
derived_columns

生成后的衍生列会保留在 `screen`，因此可以再把它们作为普通原始字段直接回测。

In [ ]:
derived_to_test = derived_columns[:5]
derived_raw_results = test_raw_variables(
    screen=screen,
    returns=returns,
    variables=derived_to_test,
    list_noire_path=list_noire_path,
    percentile=0.13,
    show_plot=True,
) if derived_to_test else {}

## 6. 增量测试

先回测一次基线，然后把每个候选变量单独加入基线。候选变量使用的 level、pct、diff 和权重完全由 `CANDIDATE_CONFIG` 决定。

In [ ]:
BASELINE_CONFIG = {
    "Quality Avg Percentile": {
        "higher_is_better": True,
        "use_level": True, "weight_level": 1.0,
        "use_pct": False, "weight_pct": 0.0,
        "use_diff": False, "weight_diff": 0.0,
    },
}

CANDIDATE_CONFIG = available_config

incremental_results = test_incremental_signals(
    screen=screen,
    returns=returns,
    baseline_config=BASELINE_CONFIG,
    candidate_config=CANDIDATE_CONFIG,
    list_noire_path=list_noire_path,
    percentile=0.13,
    show_plot=True,
)

In [ ]:
incremental_summary = pd.DataFrame([
    {
        "Test": name,
        "Robust Score": payload["backtest"].get("robust_score"),
        "Top/Bench": payload["backtest"].get("top_bench_ratio"),
        "Top/Worst": payload["backtest"].get("top_worst_ratio"),
    }
    for name, payload in incremental_results.items()
]).sort_values("Robust Score", ascending=False)
incremental_summary

## 7. 构建并回测宽口径综合分数

这一步严格使用 `WIDE_CONFIG` 中启用的维度及权重。生成的 `Score_Wide` 和相关衍生字段会直接加入 `screen`。结果对象同时直接提供 `raw_variables`、`raw_variable_weights` 和逐维度的 `composition`。

In [ ]:
wide_results = test_composite_signal(
    screen=screen,
    returns=returns,
    score_col="Score_Wide",
    signal_config=available_config,
    list_noire_path=list_noire_path,
    percentile=0.13,
    show_plot=True,
)
screen = wide_results["screen"]
wide_result = wide_results["backtest"]

In [ ]:
display(pd.Series({
    "Robust Score": wide_result.get("robust_score"),
    "Top/Bench": wide_result.get("top_bench_ratio"),
    "Top/Worst": wide_result.get("top_worst_ratio"),
}, name="Score_Wide"))

print("Raw variables:", wide_results["raw_variables"])
display(pd.DataFrame.from_dict(
    wide_results["raw_variable_weights"], orient="index"
))
display(wide_results["composition"])

## 8. 集中比较与导出

将不同类型的结果放进同一个字典。`compare_backtest_results()` 会生成可排序的总表，并同时展示每个测试的原始变量、维度、方向和权重配方。

建议先在同一类测试内部比较 `robust_rank_within_type`，再检查 Robust Score 的分解项，最后结合 Top/Bench、Top/Worst 曲线判断。增量候选还会自动生成 `*_delta_vs_baseline` 字段，直接显示相对基线改善了多少。

In [ ]:
ALL_RESULTS = {
    "raw": raw_results,
    "unitary": unitary_results,
    "derived_raw": derived_raw_results,
    "incremental": incremental_results,
    "wide": {"Score_Wide": wide_results},
}

comparison_table = compare_backtest_results(ALL_RESULTS)
comparison_table

In [ ]:
classic_view = comparison_table[[
    "test_type", "test_name", "robust_score",
    "top_total_return", "top_annualized_return",
    "top_annualized_volatility", "top_sharpe_ratio",
    "top_max_drawdown", "top_information_ratio",
    "top_beta", "top_tracking_error", "top_sortino_ratio",
]].sort_values("robust_score", ascending=False)
classic_view

导出目录包含：

- `backtest_summary.csv`：所有回测指标、排名、raw variables、权重摘要和可读配方；
- `signal_composition.csv`：一行一个组成变量及维度权重，同时包含 raw variable 总权重和绝对权重占比；
- `classic_metrics.csv`：Top、Worst、Benchmark 的经典指标长表；
- `period_metrics.csv`：每个测试在每个时期的 CAGR、经典指标及时期内排名；
- `metrics_by_period.csv`：把 Total period 与所有断点时期合并为统一的测试 × 时期总表；
- `backtest_registry.json`：完整机器可读登记册；
- `figures/`：按 `export_html` 和 `export_png` 开关保存 HTML 或 PNG 图；
- `data/`：每个测试的净值与比率曲线；
- `holdings/`：每个测试的 Top/Worst 历史持仓。

所有数据文件和总表会先于图片写入。PNG 导出依赖 Kaleido；如果不需要图片，建议同时关闭 `export_html` 和 `export_png`。

In [ ]:
# Décommentez cette ligne une seule fois si l'export PNG est indisponible.
# %pip install --upgrade kaleido

In [ ]:
export_bundle = export_backtest_results(
    results=ALL_RESULTS,
    output_dir=PLUGIN_DIR / "exports",
    export_name="factor_research_run",
    export_html=False,
    export_png=False,
)

export_bundle["export_dir"]

### 重构和组合 performance

`combine_backtest_performances()` 优先使用内存中的 `ALL_RESULTS`，并用导出目录中的 `data/*_performance.csv` 补充缺失结果。Kernel 重启后，`globals().get("ALL_RESULTS")` 会返回 `None`，函数会自动完全从本地文件恢复。

`reconstruct_backtest_analysis()` 会一次恢复 performance、composition、Total period 的所有 scores/metrics、经典指标长表以及所有分时期指标。`metrics_by_period` 会把 `Période totale` 和每个断点时期合并为一张“测试 × 时期”总表，适合筛选和透视。Kernel 重启后会从正式 CSV 自动恢复；内存结果存在时则直接重建。Composite 的自动列名会直接列出全部 raw variables、启用的维度及权重，例如 `composite / Revenue 5Y CAGR[level×1.0, pct×0.5] | FCF Conversion[level×1.0, diff×0.5] | Net Debt to Ebit[level×1.0, diff×0.5] | Top`。Ratio 由 `calculate_performance_ratios()` 独立计算，`plot_performance_comparison()` 只负责用 Plotly 绘制上下两个面板。

In [ ]:
EXPORT_DIR = PLUGIN_DIR / "exports" / "factor_research_run"

analysis_bundle = reconstruct_backtest_analysis(
    results=globals().get("ALL_RESULTS"),
    export_dir=EXPORT_DIR,
    portfolios=("Top",),
    performance_save_path=EXPORT_DIR / "all_top_performance.csv",
)

all_top_performance = analysis_bundle["performance"]
performance_composition = analysis_bundle["performance_composition"]
backtest_summary = analysis_bundle["summary"]
classic_metrics = analysis_bundle["classic_metrics"]
period_metrics = analysis_bundle["period_metrics"]
metrics_by_period = analysis_bundle["metrics_by_period"]
signal_composition = analysis_bundle["signal_composition"]

composite_composition = performance_composition.loc[
    performance_composition["test_type"].eq("composite"),
    [
        "display_path", "raw_variable", "dimension",
        "higher_is_better", "weight", "denominator",
        "raw_variable_total_weight",
        "raw_variable_absolute_weight_share",
    ],
]

display(all_top_performance)
display(composite_composition)
display(backtest_summary)
display(classic_metrics)
display(metrics_by_period)

In [ ]:
COMPOSITE_TEST_PATH = performance_composition.loc[
    performance_composition["test_type"].eq("composite"), "test_path"
].drop_duplicates().iloc[0]

PERFORMANCE_SELECTION = {
    "Raw Revenue Top": ("raw / Revenue 5Y CAGR", "Top"),
    "Unitary Revenue Level Top": ("unitary / Revenue 5Y CAGR | level", "Top"),
    "Incremental FCF Top": ("incremental / FCF Conversion", "Top"),
    "Composite Top": (COMPOSITE_TEST_PATH, "Top"),
    "Benchmark": (COMPOSITE_TEST_PATH, "Bench"),
}

selected_performance = combine_backtest_performances(
    results=globals().get("ALL_RESULTS"),
    export_dir=EXPORT_DIR,
    selections=PERFORMANCE_SELECTION,
    save_path=EXPORT_DIR / "selected_performance_comparison.csv",
)

selected_ratios = calculate_performance_ratios(
    selected_performance,
    benchmark_column="Benchmark",
)

comparison_figure = plot_performance_comparison(
    performance=selected_performance,
    ratios=selected_ratios,
    benchmark_column="Benchmark",
    title="Comparaison des performances",
    save_path=None,
    show_plot=True,
)

## 9. 分时期比较

所有回测入口只使用一个时期设置：断点年份列表 `period_breakpoints`。默认推荐值为 `[2020, 2022, 2024]`，会把时间线切成 2020 年以前、2020–2021、2022–2023、2024 至今四段。每张交互式 Plotly 图顶部都有时期下拉菜单，切换后标题会显示该时期的 Top CAGR、主动 CAGR 和 Top/Worst CAGR。

`backtest_summary.csv` 中包含 `period_<时期>_<指标>` 形式的关键宽表字段；`period_metrics.csv` 则是一行一个“测试 × 时期”的完整明细，适合筛选和透视。

In [ ]:
pd.DataFrame(build_periods_from_breakpoints(RECOMMENDED_PERIOD_BREAKPOINTS))

In [ ]:
period_table = period_metrics
period_view = period_table[[
    "period_label", "test_type", "test_name",
    "top_cagr", "active_cagr", "top_worst_cagr",
    "top_sharpe_ratio", "top_max_drawdown",
    "active_cagr_rank_global", "active_cagr_rank_within_type",
]]
period_view.sort_values(["period_label", "active_cagr_rank_global"])

In [ ]:
active_cagr_pivot = period_table.pivot_table(
    index=["test_type", "test_name"],
    columns="period_id",
    values="active_cagr",
)
active_cagr_pivot.sort_values(active_cagr_pivot.columns[-1], ascending=False)

断点年份表示新时期的起点，因此 `[2009, 2020]` 会生成 2009 年以前、2009–2019、2020 至今三段。代码中推荐的断点是 2020、2022、2024；如果数据覆盖 2009 年以前，也可以加入 2009。传入空列表 `[]` 时只保留一个全样本时期。

In [ ]:
PERIOD_BREAKPOINTS = [2009, 2020]
display(pd.DataFrame(build_periods_from_breakpoints(PERIOD_BREAKPOINTS)))

# Exemple d'appel direct avec les points de rupture.
# breakpoint_results = test_raw_variables(
#     screen, returns, RAW_VARIABLES, list_noire_path,
#     period_breakpoints=PERIOD_BREAKPOINTS,
# )

## 10. 结果对象怎么读取

普通回测结果包含：

```python
result["robust_score"]
result["top_bench_ratio"]
result["top_worst_ratio"]
result["performance"]
result["ratios"]
result["figure"]
result["top_holdings"]
result["worst_holdings"]
result["period_metrics"]
```

增量测试多一层 `backtest`：

```python
incremental_results["Net Debt to Ebit"]["backtest"]["robust_score"]
```

Robust Score 需要至少 756 个共同交易日；历史不足时会返回 `NaN`。

内部流程已经解耦：`calculate_top_vs_bottom_results()` 先生成 performance、ratios 和所有 metrics，`plot_top_vs_bottom_results()` 再消费这个结果对象绘图。批量运行且只需要数据时，可以在任一测试入口中同时设置 `show_plot=False, build_figure=False`；之后仍可通过本地 performance CSV 重构比较图。